# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIRˆ2) Exploration with `mlcroissant`

This notebook provides an example for loading and exploring the FAIRˆ2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following the Croissant schema specification.

### Dataset Source

The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Number of record sets: {len(metadata.record_sets)}")


## 2. Data Overview

Review available record sets, fields, and their IDs. Each entity is referenced via its `@id`.

In [ ]:
# List all record sets and their fields by their @id
if not metadata.record_sets:
    print("No record sets found in metadata.")
else:
    for rs in metadata.record_sets:
        print(f"RecordSet: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            for f in rs.fields:
                field_id = getattr(f, 'id', None)
                field_name = getattr(f, 'name', None)
                print(f"  Field: {field_id} (name: {field_name})")
        else:
            print("  No fields defined for this record set.")


## 3. Data Extraction

Load data from the main record set(s) into DataFrames for analysis. **All references use their `@id` fields.** To identify the correct record set, check the output above—often only one principal record set per dataset.


In [ ]:
# Extract data from all available record sets by @id
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records from RecordSet {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"\n[Columns in '{record_set_id}']:")
        print(dataframes[record_set_id].columns.tolist())
        print(f"\nPreview of '{record_set_id}':")
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for record set {record_set_id}.")

# For further operations, use the first (main) record set
if record_sets:
    main_record_set = record_sets[0]
else:
    main_record_set = None

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. Ensure all fields are referenced by their `@id`. (Field IDs can be seen in the Data Extraction output.)


In [ ]:
import numpy as np

if not main_record_set or main_record_set not in dataframes:
    print("No main record set available for EDA.")
else:
    df = dataframes[main_record_set]
    print(f"Shape of DataFrame: {df.shape}")
    
    # List numeric fields (float/int) by inspecting types or by field name heuristics
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print("Numeric fields (by type):", numeric_candidates)
    
    # For demonstration, pick the first numeric field (or set explicitly)
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        numeric_field = None
        print("No numeric field found for EDA.")
    
    # Attempt typical EDA: filter and normalize
    if numeric_field:
        threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (records: {len(filtered_df)}):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() or 1)
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Find a potential group field (categorical)
        cat_candidates = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < df.shape[0]//2]
        if cat_candidates:
            group_field = cat_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped means of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print("Unable to perform numeric analysis (no numeric field found).")

## 5. Visualization

Visualize a numeric field's distribution and the relationship to a categorical variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set and numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15, color='cornflowerblue')
    plt.title(f"Distribution of {numeric_field} in record set {main_record_set}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group_field if available
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIRˆ2 clinical dataset published as a Croissant package, using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library. We examined record sets, referenced by their `@id`, loaded data into DataFrames, filtered and normalized a numeric field, grouped by categorical variables, and visualized distributions. This flow enables rapid, reproducible machine learning and data science workflows for curated, FAIR clinical datasets.